In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# อ่านข้อมูล
df = pd.read_excel("/content/new data.xlsx", sheet_name="Sheet2")  # เปลี่ยนชื่อไฟล์ตามจริง

In [3]:
# ลบคอลัมน์ Date หากมี (เพราะไม่สามารถใช้กับโมเดลโดยตรงได้)
df = df.drop(columns=["Date"], errors='ignore')

In [4]:
# แยก features และ target
# Drop rows where the target variable 'dBZ' is missing
df_cleaned = df.dropna(subset=["dBZ"])
X = df_cleaned.drop(columns=["dBZ"])
y = df_cleaned["dBZ"]

In [5]:
# ฟังก์ชันวัด MAE RMSE
def evaluate_model(X_imputed, y):
    X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = RandomForestRegressor(random_state=42)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse) # Calculate RMSE from MSE
    return mae, rmse

In [6]:
# ----------- วิธีที่ 1: เติมค่าด้วย KNN -----------
# Replace '-' with NaN
X = X.replace('-', np.nan)
knn_imputer = KNNImputer(n_neighbors=5)
X_knn = knn_imputer.fit_transform(X)
mae_knn, rmse_knn = evaluate_model(X_knn, y)

/tmp/ipython-input-708479127.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = X.replace('-', np.nan)


In [7]:
# ----------- วิธีที่ 2: เติมค่าด้วย Random Forest -----------
rf_imputer = IterativeImputer(estimator=RandomForestRegressor(random_state=42), random_state=42)
X_rf = rf_imputer.fit_transform(X)
mae_rf, rmse_rf = evaluate_model(X_rf, y)

In [9]:
# แสดงผลลัพธ์
print(f"KNN Imputer:     MAE = {mae_knn:.2f}, RMSE = {rmse_knn:.2f}")
print(f"RF Imputer:      MAE = {mae_rf:.2f}, RMSE = {rmse_rf:.2f}")

KNN Imputer:     MAE = 16.05, RMSE = 18.69
RF Imputer:      MAE = 15.80, RMSE = 18.87


In [11]:
# ตรวจสอบตัวอย่าง
print("KNN Imputed Data")
print(pd.DataFrame(X_knn, columns=X.columns).head())
print("\nRF Imputed Data")
print(pd.DataFrame(X_rf, columns=X.columns).head())

KNN Imputed Data
   Showalter  Lifted   SWEAT     K  Cross  Vertical  TT Totals   CAPE    CIN  \
0      13.29   16.28  244.92 -27.1    5.7      19.7       25.4   0.00   0.00   
1      10.78   11.77  127.18 -22.1   10.3      19.3       29.6   0.00   0.00   
2       3.53    4.54  362.30 -14.3   19.3      19.7       39.0  15.68 -46.73   
3       4.66    5.24  200.40 -18.0   18.8      19.3       38.1   0.00 -50.25   
4       7.09    8.06  170.99  -0.7   15.3      18.3       33.6   0.00   0.00   

    BRN  LCL Temp   LCL P  LCL P.1   MML θ  MML q  Thickness   PWAT  
0  0.00    276.14  783.20   314.16  296.13   6.16     5749.0  12.64  
1  0.00    281.45  839.90   319.96  295.87   8.30     5746.0  16.33  
2  0.86    288.03  903.90   331.01  296.49  11.91     5729.0  25.68  
3  0.00    285.99  879.04   327.88  296.74  10.71     5734.0  22.45  
4  0.00    285.51  868.80   327.84  297.24  10.51     5733.0  22.85  

RF Imputed Data
   Showalter  Lifted   SWEAT     K  Cross  Vertical  TT Totals   

In [13]:
# บันทึก KNN Imputed
pd.DataFrame(X_knn, columns=X.columns).to_excel("data_knn_imputed.xlsx", index=False)

# บันทึก RF Imputed
pd.DataFrame(X_rf, columns=X.columns).to_excel("data_rf_imputed.xlsx", index=False)

print("บันทึกไฟล์ Excel เสร็จเรียบร้อย ✅")

บันทึกไฟล์ Excel เสร็จเรียบร้อย ✅
